# MSAF-DenseNet201 — Groundnut Leaf Disease Detection
### Runnable demonstration

Proposed CNN: DenseNet201 backbone + multi-scale feature extraction (dense blocks 2–4)
+ CBAM attention at each scale + learned attention pooling.

**Test-set result: 91.50% accuracy, Cohen's Kappa 0.8877** on a held-out
set of 918 images — ranked 1st of 30 architectures evaluated under an identical protocol.

---

**To run:** *Runtime → Change runtime type → T4 GPU* (CPU also works, just slower),
then run the cells in order. Cell 2 downloads the trained weights (`msaf_best.pt`, 163 MB)
from the GitHub Release automatically.

This notebook is self-contained — it needs no other file from the project.

## Cell 1 — Model definition
This is the proposed architecture, identical to `src/model.py`.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models

IMG_SIZE = 256
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class ChannelAttention(nn.Module):
    """CBAM channel branch - learns WHICH feature detectors matter."""
    def __init__(self, ch, r=16):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(ch, max(ch // r, 8)), nn.ReLU(inplace=False),
                                 nn.Linear(max(ch // r, 8), ch))
    def forward(self, x):
        b, ch, _, _ = x.shape
        a = self.mlp(F.adaptive_avg_pool2d(x, 1).flatten(1)) + \
            self.mlp(F.adaptive_max_pool2d(x, 1).flatten(1))
        return x * torch.sigmoid(a).view(b, ch, 1, 1)


class SpatialAttention(nn.Module):
    """CBAM spatial branch - learns WHERE on the leaf they fire."""
    def __init__(self, k=7):
        super().__init__(); self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        s = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)
        return x * torch.sigmoid(self.conv(s))


class CBAM(nn.Module):
    def __init__(self, ch):
        super().__init__(); self.ca, self.sa = ChannelAttention(ch), SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))


class AttnPool(nn.Module):
    """Learned spatial pooling. Replaces global average pooling, which would
    dilute a few lesion cells against dozens of healthy-tissue cells."""
    def __init__(self, ch):
        super().__init__(); self.score = nn.Conv2d(ch, 1, 1)
    def forward(self, x):
        b, ch, h, w = x.shape
        a = torch.softmax(self.score(x).view(b, 1, h * w), dim=-1)
        att = (x.view(b, ch, h * w) * a).sum(-1)
        gap = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return torch.cat([att, gap], 1)


class ScaleBranch(nn.Module):
    """One scale: 1x1 projection -> CBAM -> attention pooling."""
    def __init__(self, in_ch, proj=256):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(in_ch, proj, 1, bias=False),
                                  nn.BatchNorm2d(proj), nn.SiLU(inplace=True))
        self.cbam = CBAM(proj); self.pool = AttnPool(proj); self.out_dim = proj * 2
    def forward(self, x): return self.pool(self.cbam(self.proj(x)))


class MSAF_DenseNet201(nn.Module):
    """Proposed CNN - multi-scale attention fusion over a DenseNet201 backbone."""
    def __init__(self, ncls, proj=256, p=0.4, pretrained=False):
        super().__init__()
        w = models.DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None
        f = models.densenet201(weights=w).features
        self.stem   = nn.Sequential(f.conv0, f.norm0, f.relu0, f.pool0,
                                    f.denseblock1, f.transition1)
        self.block2 = f.denseblock2; self.trans2 = f.transition2   # -> stride 8
        self.block3 = f.denseblock3; self.trans3 = f.transition3   # -> stride 16
        self.block4 = nn.Sequential(f.denseblock4, f.norm5)        # -> stride 32
        self.s3 = ScaleBranch(512,  proj)
        self.s4 = ScaleBranch(1792, proj)
        self.s5 = ScaleBranch(1920, proj)
        d = self.s3.out_dim + self.s4.out_dim + self.s5.out_dim
        self.head = nn.Sequential(
            nn.BatchNorm1d(d), nn.Dropout(p),
            nn.Linear(d, 512), nn.BatchNorm1d(512), nn.SiLU(inplace=True),
            nn.Dropout(p * 0.5), nn.Linear(512, ncls))

    def features(self, x):
        """The three scales tapped from the backbone."""
        x  = self.stem(x)
        c3 = F.relu(self.block2(x), inplace=False)
        c4 = F.relu(self.block3(self.trans2(c3)), inplace=False)
        c5 = F.relu(self.block4(self.trans3(c4)), inplace=False)
        return c3, c4, c5

    def classify(self, c3, c4, c5):
        return self.head(torch.cat([self.s3(c3), self.s4(c4), self.s5(c5)], 1))

    def forward(self, x):
        return self.classify(*self.features(x))


print("architecture defined")

## Cell 2 — Load the trained weights
Downloaded from the GitHub Release (163 MB, about a minute). If the download fails, you are asked to upload `msaf_best.pt` instead.

In [ ]:
import os, torch
from google.colab import files

CKPT = "msaf_best.pt"
URL  = "https://github.com/SidMalik2005/Groundnut-leaf-disease-MSAF/releases/latest/download/msaf_best.pt"
if not os.path.exists(CKPT):
    os.system(f"wget -q --show-progress -O {CKPT} {URL}")
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 1_000_000:
    print("Download failed - upload msaf_best.pt:")
    up = files.upload()
    CKPT = list(up.keys())[0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ck = torch.load(CKPT, map_location="cpu", weights_only=False)
CLASSES = ck["classes"]
net = MSAF_DenseNet201(len(CLASSES))
net.load_state_dict(ck["ema"] if ck.get("which") == "ema" else ck["model"])
net.to(device).eval()

print(f"\nloaded on {device}")
print(f"classes ({len(CLASSES)}): {CLASSES}")
print(f"parameters: {sum(p.numel() for p in net.parameters())/1e6:.2f} M")
print(f"validation accuracy at checkpoint: {ck.get('acc', 0)*100:.2f}%  (epoch {ck.get('epoch',0)+1})")

## Cell 3 — Inference + Grad-CAM helpers

In [ ]:
import numpy as np, torch.nn.functional as F
from PIL import Image
from torchvision import transforms

tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
PRETTY = {c: c.replace("_", " ").title() for c in CLASSES}


def predict(x, tta=True):
    """4-way flip test-time augmentation, exactly as used for the reported metrics."""
    with torch.no_grad():
        p = torch.softmax(net(x).float(), 1)
        if tta:
            p = (p + torch.softmax(net(torch.flip(x, [3])).float(), 1)
                   + torch.softmax(net(torch.flip(x, [2])).float(), 1)
                   + torch.softmax(net(torch.flip(x, [2, 3])).float(), 1)) / 4
    return p[0].cpu().numpy()


def grad_cam(x, idx):
    """Grad-CAM over dense block 4. Needs gradients, so no torch.no_grad() here."""
    net.zero_grad(set_to_none=True)
    c3, c4, c5 = net.features(x)
    c5.retain_grad()
    net.classify(c3, c4, c5)[0, idx].backward()
    w   = c5.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * c5).sum(1, keepdim=True))
    cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
    cam = cam[0, 0].detach().float().cpu().numpy()
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)


def analyse(path):
    img   = Image.open(path).convert("RGB")
    x     = tf(img).unsqueeze(0).to(device)
    probs = predict(x)
    top   = int(probs.argmax())
    return img, probs, top, grad_cam(x, top)


print("ready")

## Cell 4 — Run it on a leaf image
Upload any groundnut leaf photograph when prompted.

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files

up   = files.upload()
path = list(up.keys())[0]

img, probs, top, cam = analyse(path)

print("\n" + "=" * 52)
print(f"  PREDICTION:  {PRETTY[CLASSES[top]]}   ({probs[top]*100:.2f}%)")
print("=" * 52)
for i in np.argsort(-probs):
    bar = "#" * int(probs[i] * 40)
    print(f"  {PRETTY[CLASSES[i]]:<22} {probs[i]*100:6.2f}%  {bar}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
ax[0].imshow(img.resize((IMG_SIZE, IMG_SIZE))); ax[0].set_title("Input"); ax[0].axis("off")
ax[1].imshow(img.resize((IMG_SIZE, IMG_SIZE))); ax[1].imshow(cam, cmap="jet", alpha=0.45)
ax[1].set_title("Grad-CAM — evidence the model used"); ax[1].axis("off")
order = np.argsort(probs)
ax[2].barh([PRETTY[CLASSES[i]] for i in order], [probs[i]*100 for i in order],
           color=["#1b5e20" if i == top else "#c8d6c8" for i in order])
ax[2].set_xlim(0, 105); ax[2].set_xlabel("confidence (%)")
for sp in ax[2].spines.values(): sp.set_visible(False)
ax[2].set_title(f"{PRETTY[CLASSES[top]]} — {probs[top]*100:.1f}%",
                color="#1b5e20", fontweight="bold")
plt.tight_layout(); plt.savefig("prediction.png", dpi=170); plt.show()

## Cell 5 — Optional: several images at once
Upload multiple leaf photographs and get one summary figure.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from google.colab import files

up = files.upload()
paths = list(up.keys())
n = len(paths)

fig = plt.figure(figsize=(13, 3.1 * n))
gs  = GridSpec(n, 3, figure=fig, width_ratios=[1, 1, 1.5], hspace=0.4, wspace=0.3)
for r, pth in enumerate(paths):
    img, probs, top, cam = analyse(pth)
    a0, a1, a2 = [fig.add_subplot(gs[r, k]) for k in range(3)]
    a0.imshow(img.resize((IMG_SIZE, IMG_SIZE))); a0.axis("off")
    a0.set_title(pth[:28], fontsize=9)
    a1.imshow(img.resize((IMG_SIZE, IMG_SIZE))); a1.imshow(cam, cmap="jet", alpha=0.45)
    a1.axis("off"); a1.set_title("Grad-CAM", fontsize=9)
    order = np.argsort(probs)
    a2.barh([PRETTY[CLASSES[i]] for i in order], [probs[i]*100 for i in order],
            color=["#1b5e20" if i == top else "#c8d6c8" for i in order])
    a2.set_xlim(0, 105); a2.tick_params(labelsize=8)
    for sp in a2.spines.values(): sp.set_visible(False)
    a2.set_title(f"{PRETTY[CLASSES[top]]} — {probs[top]*100:.1f}%",
                 fontsize=9, color="#1b5e20", fontweight="bold")
    print(f"{pth:<34} -> {CLASSES[top]:<22} {probs[top]*100:6.2f}%")

plt.savefig("batch_predictions.png", dpi=150, bbox_inches="tight"); plt.show()

---

### The full web application

This notebook runs the model directly. The browser interface — drag-and-drop upload,
confidence bars, Grad-CAM panel, reference notes, and a JSON API — is in the project
archive (`app/app.py` + `app/templates/index.html`). To run it locally see `SETUP.md`.

### Where the contribution lives

`src/model.py`, specifically `ScaleBranch`, `CBAM`, `AttnPool` and
`MSAF_DenseNet201.features()`. A standard DenseNet201 hands only its final 8×8
feature map to the classifier and then global-average-pools it; both steps destroy
the evidence for early leaf spot, which is a handful of small lesions on an
otherwise green leaf. The three modifications address exactly that, and early leaf
spot recall rose from 81.51% on the DenseNet201 baseline to 91.70%.